# My glacier — tutorial (Session 2)

**Follow this notebook from top to bottom**: it guides you through the mini-project step by step. It lives in **your own copy** of the template, `~/work/my-glacier/` (copied once from `~/work/glamod/session2/my-glacier/`). It only **reads and plots** the results of your IGM runs: re-run it as often as you like.

| What | Where |
|---|---|
| **Set** the parameters | YAML files in `experiment/` (text editor) |
| **Run** IGM | a **Terminal**, from this folder: `cd ~/work/my-glacier` |
| **Look** at the results | this notebook, reading `outputs/<run>/` |

Steps:
0. Put the RGI ID of your glacier (`RGI_ID`) in each file of `experiment/`
1. **Present**: download the data, check the initial state
2. **Calibration of the ice-flow parameters** (`arrhenius`, `tau_ref`) on the observed speeds
3. **Past**: Little Ice Age → today, **calibration of the SMB** (`melt_enhancer`)
4. **Future**: climate scenarios 2020–2100
5. **A simple SMB model** (ELA-based)
6. **Response time**

Session 3 (moraines, figures for the report): `tutorial-session3.ipynb`.

In [ ]:
import sys, os, re, yaml
sys.path.insert(0, os.path.expanduser("~/work/glamod/utils"))   # course plotting helpers (utils/helper.py)
sys.path.insert(0, os.path.abspath("../../utils"))               # (same, when run inside the course folder)
from helper import *

with open("experiment/params_past.yaml") as f:
    RGI_ID = yaml.safe_load(f)["inputs"]["oggm_shop"]["RGI_ID"]
INPUT = f"data/input_{RGI_ID}.nc"
print("My glacier:", RGI_ID)
os.makedirs("figures", exist_ok=True)                  # figures for the report go here

## 1. Present: download the data and check the initial state

▶ **In the terminal**: a 1-year run downloads the data (≈ 1 min the first time):

```bash
igm_run +experiment=params_present
```

In [ ]:
plot_input_data(INPUT)
plot_climate(RGI_ID)

## 2. Calibration of the ice-flow parameters

First, compare the speeds modelled with the default parameters (`arrhenius` = 78, `tau_ref` = 0.215) with the observed ones (Millan et al., 2022):

In [ ]:
compare_modeled_observed_velocity(INPUT, "outputs/params_present/output.nc", year=2020)

Then explore **all combinations** of ice softness (`arrhenius`) and basal sliding (`tau_ref`) with 1-year runs. `-m` (multirun) makes one run per combination of the values given in the command line (3 × 4 = 12 runs, ≈ 3 min).

▶ **In the terminal**:

```bash
igm_run -m +experiment=params_sensitivity processes.iceflow.physics.viscosity.arrhenius=25,50,100 processes.iceflow.physics.sliding.tau_ref=0.1,0.2,0.4,0.8
```

Results: `outputs/params_sensitivity/0`, `1`, …, `11`. Each map below shows the speed difference (modelled − observed) of one run: look for a **bias close to 0** and a small RMSE (in red). Refine the values around the best one if needed (re-run with other values: the folder is overwritten).

In [ ]:
runs, labels = sweep_runs("outputs/params_sensitivity")   # the 12 runs
compare_velocity_misfit(INPUT, runs, year=2020)

> ✅ **Calibration 1 — ice flow** (double-click to edit): `arrhenius` = ...  `tau_ref` = ...
>
> (example, Rhone Glacier: `arrhenius` = 30, `tau_ref` = 0.315)
>
> ➡️ Put these values in the `iceflow → physics` block of **all** the files of `experiment/`.

## 3. Past: from the Little Ice Age to today — calibration of the SMB

Start in 1700 from today's geometry, advance to the LIA maximum (~1850), retreat with the observed climate 1901–2019 (≈ a few minutes). Keep the pre-1901 climate (`clim_trend_array`) of `experiment/params_past.yaml` and **adjust `melt_enhancer`** (start with 1.0, explore 0.5–2) until the glacier (i) reaches the LIA moraines around 1850 (Siegfried map on map.geo.admin.ch) and (ii) matches today's area and volume in 2020.

▶ **In the terminal**:

```bash
igm_run +experiment=params_past
```

To keep several tries, copy the file (e.g. `params_past_v2.yaml`, and change its run folder at the end of the file to `outputs/params_past_v2`) and add the run to `past_runs` below.

In [ ]:
run = "outputs/params_past"
show_dashboard(run, n=4)
analyze_time_series(os.path.join(run, "output_ts.nc"))

In [ ]:
# Ice thickness through time, and along the main flowline
animate_glacier_evolution(os.path.join(run, "output.nc"), variable="thk")
plot_flowline_evolution(os.path.join(run, "output.nc"), rgi_id=RGI_ID)

In [ ]:
# Compare several past runs with today's observed area / volume (from the input data)
ds = xr.open_dataset(INPUT); dx = float(ds.x[1] - ds.x[0])
obs = {"area": ([2020], [float((ds.thk > 1).sum()) * dx**2 / 1e6]),
       "vol":  ([2020], [float(ds.thk.sum()) * dx**2 / 1e9])}
past_runs = ["outputs/params_past"]              # <- add your other past runs here
compare_runs(past_runs, [os.path.basename(r) for r in past_runs], observations=obs)

> ✅ **Calibration 2 — SMB** (double-click to edit): `melt_enhancer` = ...
>
> ➡️ Put this value in `params_future.yaml` (and `params_past_moraines.yaml` for Session 3).

## 4. Future: climate scenarios 2020–2100

`delta_temp` is relative to **1960–1990** (see Lecture 2):
- 2020 row (in `experiment/params_future.yaml`): present-day offset (2000–2019, printed by `plot_climate` above)
- 2100: 1981–2010 offset + CH2018 warming of the scenario, given **in the command line**, one value per scenario (committed = today's offset, RCP2.6, RCP4.5, RCP8.5)

▶ Check that `params_future.yaml` contains your calibrations 1 and 2, then **in the terminal** (4 runs, a few minutes; values here for the Rhone Glacier, adapt them):

```bash
igm_run -m +experiment=params_future processes.climate.oggm.clim_trend_array.2.1=1.3,2.0,3.4,5.1
```

One run per scenario: `outputs/params_future/0`, `1`, `2`, `3` (in the order of the list).

In [ ]:
runs, labels = sweep_runs("outputs/params_future")    # the scenario runs, labelled by their 2100 delta_temp
compare_runs(runs, labels)

In [ ]:
# Maps for one scenario (here the third one: RCP4.5)
animate_glacier_evolution(runs[2] + "/output.nc", variable="thk")

## 5. A simple SMB model: ELA-based

Instead of the climate, prescribe the ELA history yourself. **Try to reproduce the volume curve of your calibrated past run** by playing with the ELAs (and gradients) in `experiment/params_past_simple.yaml` (today's ELA: end-of-summer snowline, GLAMOS).

▶ **In the terminal**:

```bash
igm_run +experiment=params_past_simple
```

In [ ]:
plot_smb_simple(ela=3000, gradabl=0.009, gradacc=0.005, accmax=2.0)   # <- your ELA and gradients
compare_runs(["outputs/params_past", "outputs/params_past_simple"], ["climate-driven SMB", "ELA-based SMB"], observations=obs)

## 6. Response time (your own IGM module, oscillating ELA)

▶ **In the terminal** (400 model years, a few minutes). For another period, copy `experiment/params_response_time.yaml` to `experiment/params_response_100.yaml`, set `period: 100` and the run folder `outputs/params_response_100` in it:

```bash
igm_run +experiment=params_response_time
igm_run +experiment=params_response_100
```

In [ ]:
plot_response_time("outputs/params_response_time")
# plot_response_time("outputs/params_response_100")